# 02. Context managers

**Tier:** Intermediate. **Why you'd use this:** Opening a door should always log a close, even if the body errors. A context manager pairs setup with guaranteed cleanup.

**After this lesson you can:** write `__enter__` and `__exit__`; rely on cleanup after errors; know what `__exit__` return values mean.
About 25 minutes. **Uses:** [Dunder methods](01-dunder-lesson.ipynb), [Else and finally](../06-errors/04-else-finally-lesson.ipynb)


## Quick review

1. What happens to `__hash__` when you define `__eq__`?
2. How does an empty `__len__` affect truthiness?

<details><summary>Answers</summary>

It becomes `None` unless you define a compatible hash. The instance is false in boolean context.
</details>


## The idea

Objects that implement `__enter__` and `__exit__` work with `with`. `__enter__` runs first and its return value is bound by `as`. `__exit__` always runs afterward for cleanup. Returning true from `__exit__` swallows an exception; returning false does not.

<details><summary>Explain more</summary>

`with Door() as entered:` opens, runs the body, then closes. The close happens even if the body raises.
</details>


## See it work

Log open, inside, and close.


In [1]:
class Door:
    def __enter__(self):
        # 1. Setup, then hand the object to the with-as target.
        self.log = ["open"]
        return self

    def __exit__(self, exc_type, exc, tb):
        self.log.append("close")
        return False


with Door() as entered:
    entered.log.append("inside")
print(entered.log)

['open', 'inside', 'close']


## Check yourself

Answer these before you run the cell.

1. Does `__exit__` run if the body raises?
2. What does returning `True` from `__exit__` do?


In [2]:
class Door:
    def __enter__(self):
        self.log = ["open"]
        return self

    def __exit__(self, exc_type, exc, tb):
        self.log.append("close")
        return False


try:
    with Door() as entered:
        entered.log.append("inside")
        raise RuntimeError("boom")
except RuntimeError:
    print(entered.log)

['open', 'inside', 'close']


<details><summary>Answers</summary>

Yes. It suppresses the exception so the `with` does not propagate it.
</details>


## Why it works this way

A `with` block calls `__enter__` before the body and `__exit__` after it, even when the body raises. That is the guarantee: cleanup is not skipped by an early `return` or exception. `__exit__` receives exception info. Returning a true value from `__exit__` suppresses the exception; returning false lets it propagate. Deep dive: [Attribute lookup](../../internals/08-attributes/01-lookup-lesson.ipynb).

Context managers exist because resources need pairing: open with close, lock with unlock, span with end. Manual `try`/`finally` works, but `with` keeps the pairing at the call site. The object returned by `__enter__` is what `as` binds. Often it is `self`; sometimes it is a handle.

Cost is two method calls around the body. The design reason is reliable cleanup without repeating `finally` at every call site. Evidence: a log that always appends `"close"` after `"inside"`, and an `__exit__` that still runs when the body raises.

The three-argument form of `__exit__` exists so cleanup can see what failed. Logging, rollback, and metrics often need the exception type even when they re-raise. Returning true is a sharp tool: it turns a failed body into a successful `with` statement. Use it for expected, handled failures only, and test that path explicitly.

Context managers compose. Nested `with` statements, or a single `with` with multiple managers, run enters left to right and exits right to left. That nesting is how files inside locks, or transactions inside sessions, stay ordered. Prefer `with` at the call site over relying on `__del__` for cleanup. Destructors are late and unreliable by comparison.

Think of `__enter__`/`__exit__` as a protocol for paired actions. The body is the critical section. Cleanup is not optional. When teaching or reviewing, ask what happens if the body returns early and what happens if it raises. Both must close.


In [3]:
class Door:
    def __enter__(self):
        self.log = ["open"]
        return self

    def __exit__(self, exc_type, exc, tb):
        self.log.append("close")
        return False


with Door() as entered:
    entered.log.append("inside")
print(entered.log)

try:
    with Door() as entered:
        entered.log.append("inside")
        raise RuntimeError("boom")
except RuntimeError:
    print(entered.log)

['open', 'inside', 'close']
['open', 'inside', 'close']


## When to use it

Use a context manager when setup and cleanup must pair. Return `False` from `__exit__` unless you intentionally handle the error. Prefer `with` over scattered `try`/`finally` at call sites. Don't suppress exceptions unless the manager truly handled them. If cleanup is best-effort only, say so in the docstring; otherwise treat it as guaranteed.


## Common mistake

Forgetting to return `False` explicitly is fine (None is false), but returning `True` by accident swallows real bugs.


In [4]:
class Door:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, tb):
        print("closed")
        return True


with Door():
    raise RuntimeError("boom")
print("after")

closed
after


Return `False` or nothing so the error still propagates after cleanup.


In [5]:
class Door:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, tb):
        print("closed")
        return False


try:
    with Door():
        raise RuntimeError("boom")
except RuntimeError:
    print("after")

closed
after


## Try it

Change `Door` so `__enter__` returns the log list instead of `self`. What does the `as` name bind, and who still owns `__exit__`?


## Recap

- `with` calls `__enter__` then always `__exit__`.
- Cleanup runs after success, return, and errors.
- `__exit__` returning true suppresses the exception.

**Next:** [practice](02-context-managers-practice.ipynb)
